In [ ]:
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import calendar
import os

def create_multiyear_schedule():

    print("   КЕСТЕ ГЕНЕРАТОРЫ (WALK-FORWARD)   ")

    try:
        start_year = int(input("НАЧАЛО(OOS): "))
        end_year = int(input("ОКОНЧАНИЕ (OOS): "))
        window_months = int(input("IS (в месяцах): "))
    except ValueError:
        print("[!] Қате")
        return

    os.makedirs('CSV', exist_ok=True)
    records = []
    iteration = 1
    
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            oos_start = datetime(year, month, 1)
            last_day = calendar.monthrange(year, month)[1]
            oos_end = datetime(year, month, last_day)
            
            is_end = oos_start - relativedelta(days=1)
            is_start = oos_start - relativedelta(months=window_months)
            
            records.append({
                "Iteration": iteration,
                "IS_Start": is_start.strftime('%d.%m.%Y'),
                "IS_End": is_end.strftime('%d.%m.%Y'),
                "OOS_Start": oos_start.strftime('%d.%m.%Y'),
                "OOS_End": oos_end.strftime('%d.%m.%Y')
            })
            iteration += 1
        
    df = pd.DataFrame(records)
    df.to_csv("CSV/schedule.csv", sep=";", index=False)
    
    print("\n" + "-"*85)
    header = f"{'Ит.':<4} | {'Окно обучения (IS)':^25} | {'Период торгов (OOS)':^25}"
    print(header)
    print("-" * 85)
    
    display_rows = records if len(records) <= 15 else records[:5] + [None] + records[-5:]
    
    for row in display_rows:
        if row is None:
            print(f"{'...':<4} | {'...':^25} | {'...':^25}")
            continue
        print(f"{row['Iteration']:<4} | {row['IS_Start']} - {row['IS_End']} | {row['OOS_Start']} - {row['OOS_End']}")
    
    print("-" * 85)
    print(f"БАРЛЫҒЫ: {len(df)} қайталану сақталды 'CSV/schedule.csv'")
    print("="*85 + "\n")
    
    return df

df_schedule = create_multiyear_schedule()

In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
import warnings
import os
import sys
import logging

logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

def load_settings(path):
    """Загрузка параметров стратегии из settings.csv"""
    defaults = {
        'initial_capital': 100000.0, 'max_open_pairs': 7.0,
        'p_val': 0.05, 'min_hl': 1.0, 'max_hl': 42.0,
        'min_corr': 0.5, 'liq_quantile': 0.6, 'min_beta': 0.1, 'max_beta': 5.0,
        'min_sp_vol': 0.01, 'min_zc': 12, 'sharpe_limit': 1.5,
        'r_sq_limit': 0.85, 'z_entry': 2.0, 'z_exit': 0.5, 'hl_multiplier': 2.5,
        'commission_pct': 0.001,
        'min_hurst': 0.0, 'max_hurst': 0.5,
    }
    if not os.path.exists(path):
        return defaults
    try:
        df = pd.read_csv(path, delimiter=';', index_col='Parameter')
        res = {}
        mapping = {
            'initial_capital': 'initial_capital',
            'max_open_pairs': 'max_open_pairs',
            'p_val': 'p_value_threshold',
            'min_hl': 'min_half_life',
            'max_hl': 'max_half_life',
            'min_corr': 'min_correlation',
            'liq_quantile': 'liquidity_quantile',
            'min_beta': 'min_hedge_ratio',
            'max_beta': 'max_hedge_ratio',
            'min_sp_vol': 'min_spread_vol',
            'min_zc': 'min_zero_crossings',
            'sharpe_limit': 'sharpe_ratio_threshold',
            'r_sq_limit': 'r_squared_threshold',
            'z_entry': 'z_entry',
            'z_exit': 'z_exit',
            'hl_multiplier': 'hl_multiplier',
            'commission_pct': 'commission_pct',
            'min_hurst': 'min_hurst',
            'max_hurst': 'max_hurst',
        }
        for k, csv_key in mapping.items():
            if csv_key in df.index:
                val = str(df.loc[csv_key, 'Value']).replace(',', '.')
                res[k] = float(val)
            else:
                res[k] = defaults[k]
        return res
    except:
        return defaults

def calculate_half_life(spread):
    try:
        spread_lag = spread.shift(1).dropna()
        spread_diff = spread.diff().dropna()
        common_idx = spread_lag.index.intersection(spread_diff.index)
        y = spread_diff.loc[common_idx]
        x = sm.add_constant(spread_lag.loc[common_idx])
        model = sm.OLS(y, x).fit()
        lambda_val = model.params.iloc[1]
        if lambda_val >= 0:
            return 999
        return -np.log(2) / lambda_val
    except:
        return 999

def calculate_hurst(series):

    try:
        ts = np.asarray(series, dtype=float)
        n = len(ts)
        if n < 20:
            return 0.5
        lags = [2, 4, 8, 16, 32, 64, 128]
        lags = [l for l in lags if l < n // 2]
        if len(lags) < 2:
            return 0.5
        tau = [np.std(np.subtract(ts[lag:], ts[:-lag])) for lag in lags]
        if any(t == 0 for t in tau):
            return 0.5
        slope = np.polyfit(np.log(lags), np.log(tau), 1)[0]
        return round(float(slope), 4)
    except:
        return 0.5

def calculate_sharpe_is(spread, s1, s2, beta, z_entry, z_exit=0.5, half_life=10, hl_multiplier=2.5):

    try:
        if spread.std() == 0:
            return 0.0

        exp_mean = spread.expanding(min_periods=20).mean()
        exp_std  = spread.expanding(min_periods=20).std()
        z = ((spread - exp_mean) / exp_std).fillna(0.0).replace([np.inf, -np.inf], 0.0).values

        spread_diff = spread.diff().values
        cap = (s1 + abs(beta) * s2).values

        time_limit = max(int(half_life * hl_multiplier), 1)
        pos = np.zeros(len(z))
        curr_pos = 0
        entry_i = 0
        banned = False

        for i in range(1, len(z)):
            if banned:
                if abs(z[i]) <= z_exit:
                    banned = False
                continue
            if curr_pos == 0:
                if z[i] < -z_entry:
                    curr_pos = 1
                    entry_i = i
                elif z[i] > z_entry:
                    curr_pos = -1
                    entry_i = i
            else:
                is_conv = (curr_pos == 1 and z[i] >= -z_exit) or (curr_pos == -1 and z[i] <= z_exit)
                is_time_out = (i - entry_i) > time_limit
                if is_conv:
                    curr_pos = 0
                elif is_time_out:
                    curr_pos = 0
                    banned = True
            pos[i] = curr_pos

        with np.errstate(divide='ignore', invalid='ignore'):
            ret = np.where(cap[:-1] > 0, spread_diff[1:] / cap[:-1], 0.0)

        daily_ret = pos[:-1] * ret
        daily_ret = daily_ret[~np.isnan(daily_ret) & ~np.isinf(daily_ret)]

        if (daily_ret != 0).sum() < 10:
            return 0.0

        std = daily_ret.std()
        if std == 0:
            return 0.0

        return round((daily_ret.mean() / std) * np.sqrt(252), 2)

    except:
        return 0.0

def run_is_analysis(schedule_path, sectors_cfg_path, settings_path, sectors_dir, output_file="DATA/is_results.xlsx"):
    global ALL_LOCAL_PRICES, ALL_LOCAL_VOLUMES
    if 'ALL_LOCAL_PRICES' not in globals() or ALL_LOCAL_PRICES is None:
        print("Нарық деректерін жүктеу...")
        ALL_LOCAL_PRICES = pd.read_csv('CSV/S&P500_Sectors/market_prices.csv', index_col=0, parse_dates=True)
        ALL_LOCAL_VOLUMES = pd.read_csv('CSV/S&P500_Sectors/market_volumes.csv', index_col=0, parse_dates=True)
        print(f"Загружено: {ALL_LOCAL_PRICES.shape[1]} тикер, {len(ALL_LOCAL_PRICES)} күн")

    os.makedirs('DATA', exist_ok=True)
    stg = load_settings(settings_path)
    schedule = pd.read_csv(schedule_path, delimiter=';')
    sectors_config = pd.read_csv(sectors_cfg_path, delimiter=';')

    print("="*65)
    print(f"{' СТРАТЕГИЯ ПАРАМЕТРЛЕРІ (IS) ':-^65}")
    print(f"  > Sharpe Ratio Limit:     {stg['sharpe_limit']}")
    print(f"  > Min Correlation:        {stg['min_corr']}")
    print(f"  > R-Squared Limit:        {stg['r_sq_limit']}")
    print(f"  > P-Value Coint:          {stg['p_val']}")
    print(f"  > Half-Life:              {stg['min_hl']} - {stg['max_hl']}")
    print(f"  > Hurst Exponent:         {stg['min_hurst']} - {stg['max_hurst']}")
    print(f"  > Hedge Ratio (Beta):     {stg['min_beta']} - {stg['max_beta']}")
    print(f"  > Min Zero Crossings:     {stg['min_zc']}")
    print(f"  > Liquidity Quantile:     {stg['liq_quantile']}")
    print(f"  > Z-Entry / Z-Exit:       {stg['z_entry']} / {stg['z_exit']}")
    print(f"  > HL Multiplier:          {stg['hl_multiplier']}")
    print(f"  > Commission per leg:     {stg['commission_pct']*100:.2f}%")
    print("="*65)

    all_results = []
    stats_log = []
    total_iters = len(schedule)

    for i, (idx, row) in enumerate(schedule.iterrows(), 1):
        iter_id = row['Iteration']
        start_is = datetime.strptime(row['IS_Start'], '%d.%m.%Y').strftime('%Y-%m-%d')
        end_is = datetime.strptime(row['IS_End'], '%d.%m.%Y').strftime('%Y-%m-%d')

        iter_stats = {
            "Iteration": iter_id, "Total_Tickers": 0, "After_Liquidity": 0, "Possible_Pairs": 0,
            "After_Corr": 0, "After_Hedge_Vol": 0, "After_Cointegration": 0,
            "After_R2": 0, "After_Zero_Crossings": 0, "After_Hurst": 0, "Final_Pairs": 0
        }

        sys.stdout.write(f"\rӨңдеу: Итерация {i} из {total_iters} ({(i/total_iters)*100:.0f}%) | Жұптар табылды: {len(all_results)}")
        sys.stdout.flush()

        for _, s_row in sectors_config.iterrows():
            sector_name, liq_limit = s_row['Sector'], s_row['Liquidity_Threshold']
            sector_file = os.path.join(sectors_dir, f"{sector_name}.csv")
            if not os.path.exists(sector_file):
                continue

            tickers = [str(t).strip().replace('.', '-') for t in pd.read_csv(sector_file)['Ticker'].tolist()]
            iter_stats["Total_Tickers"] += len(tickers)

            valid_cols = [t for t in tickers if t in ALL_LOCAL_PRICES.columns]

            prices = ALL_LOCAL_PRICES.loc[start_is:end_is, valid_cols]
            volumes = ALL_LOCAL_VOLUMES.loc[start_is:end_is, valid_cols]

            if prices.empty or len(prices.columns) < 2:
                continue

            prices = prices.ffill().dropna(axis=1, how='all')
            volumes = volumes.ffill().dropna(axis=1, how='all')

            sector_adtvs = {t: (prices[t] * volumes[t]).mean()
                            for t in tickers
                            if t in prices.columns and t in volumes.columns}
            if not sector_adtvs:
                continue

            adtv_series = pd.Series(sector_adtvs)
            dynamic_threshold = max(adtv_series.quantile(stg['liq_quantile']), liq_limit)
            liquid_tickers = adtv_series[adtv_series >= dynamic_threshold].index
            valid_data = prices[liquid_tickers].ffill().dropna()

            iter_stats["After_Liquidity"] += len(liquid_tickers)
            if len(liquid_tickers) < 2:
                continue

            cols = valid_data.columns
            iter_stats["Possible_Pairs"] += (len(cols) * (len(cols) - 1)) // 2

            for ic in range(len(cols)):
                for jc in range(ic + 1, len(cols)):
                    s1, s2 = valid_data[cols[ic]], valid_data[cols[jc]]

                    if s1.corr(s2) >= stg['min_corr']:
                        iter_stats["After_Corr"] += 1
                        model = sm.OLS(s1, sm.add_constant(s2)).fit()
                        r_sq = model.rsquared
                        beta, intercept = model.params.iloc[1], model.params.iloc[0]
                        spread = s1 - (beta * s2) - intercept

                        if (stg['min_beta'] < abs(beta) < stg['max_beta']) and (spread.std() > stg['min_sp_vol']):
                            iter_stats["After_Hedge_Vol"] += 1
                            try:
                                coint_pval = coint(s1, s2)[1]
                                if coint_pval < stg['p_val']:
                                    iter_stats["After_Cointegration"] += 1

                                    if r_sq >= stg['r_sq_limit']:
                                        iter_stats["After_R2"] += 1

                                        centered = spread - spread.mean()
                                        zc = ((centered.shift(1) * centered) < 0).sum()
                                        if zc >= stg['min_zc']:
                                            iter_stats["After_Zero_Crossings"] += 1
                                            hl = calculate_half_life(spread)
                                            if stg['min_hl'] < hl < stg['max_hl']:
                                                hurst = calculate_hurst(spread.values)
                                                if stg['min_hurst'] <= hurst <= stg['max_hurst']:
                                                    iter_stats["After_Hurst"] += 1
                                                    sharpe_is = calculate_sharpe_is(
                                                        spread, s1, s2, beta,
                                                        z_entry=stg['z_entry'],
                                                        z_exit=stg['z_exit'],
                                                        half_life=hl,
                                                        hl_multiplier=stg.get('hl_multiplier', 2.5)
                                                    )
                                                    if sharpe_is >= stg['sharpe_limit']:
                                                        iter_stats["Final_Pairs"] += 1
                                                        all_results.append({
                                                            "Iteration": iter_id, "Sector": sector_name,
                                                            "Stock_1": cols[ic], "Stock_2": cols[jc],
                                                            "Corr_IS": round(s1.corr(s2), 4),
                                                            "Hedge_Ratio": round(beta, 4),
                                                            "Intercept": round(intercept, 4),
                                                            "Spread_Vol": round(spread.std(), 4),
                                                            "Half_Life": round(hl, 2),
                                                            "Hurst": round(hurst, 4),
                                                            "Sharpe_IS": sharpe_is,
                                                            "Zero_Crossings": int(zc),
                                                            "P_Val_Coint": round(coint_pval, 4),
                                                            "Month_OOS": row['OOS_Start']
                                                        })
                            except:
                                continue
        stats_log.append(iter_stats)

    print("\n" + "="*65)
    if all_results:
        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            pd.DataFrame(all_results).to_excel(writer, sheet_name='Pairs_Results', index=False)
            pd.DataFrame(stats_log).to_excel(writer, sheet_name='Filtering_Stats', index=False)
        print(f"[DONE] Сапалы жұптар табылды: {len(all_results)}. Файл: {output_file}")
    else:
        print("[!] Жұптар табылмады.")

run_is_analysis(
    schedule_path="CSV/schedule.csv",
    sectors_cfg_path="CSV/sectors.csv",
    settings_path="CSV/settings.csv",
    sectors_dir="CSV/S&P500_Sectors",
    output_file="DATA/is_results.xlsx"
)

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime

ALL_LOCAL_PRICES = pd.read_csv('CSV/S&P500_Sectors/market_prices.csv', index_col=0, parse_dates=True)
ALL_LOCAL_VOLUMES = pd.read_csv('CSV/S&P500_Sectors/market_volumes.csv', index_col=0, parse_dates=True)

def load_settings(path):
    defaults = {
        'initial_capital': 100000.0, 'max_open_pairs': 7.0,
        'p_val': 0.05, 'min_hl': 1.0, 'max_hl': 42.0,
        'min_corr': 0.5, 'liq_quantile': 0.6, 'min_beta': 0.1, 'max_beta': 5.0,
        'min_sp_vol': 0.01, 'min_zc': 12, 'sharpe_limit': 1.5,
        'r_sq_limit': 0.85, 'z_entry': 2.0, 'z_exit': 0.5, 'hl_multiplier': 2.5,
        'commission_pct': 0.001,
    }
    if not os.path.exists(path):
        return defaults
    try:
        df = pd.read_csv(path, delimiter=';', index_col='Parameter')
        res = {}
        mapping = {
            'initial_capital': 'initial_capital', 'max_open_pairs': 'max_open_pairs',
            'p_val': 'p_value_threshold',
            'min_hl': 'min_half_life', 'max_hl': 'max_half_life',
            'min_corr': 'min_correlation', 'liq_quantile': 'liquidity_quantile',
            'min_beta': 'min_hedge_ratio', 'max_beta': 'max_hedge_ratio',
            'min_sp_vol': 'min_spread_vol', 'min_zc': 'min_zero_crossings',
            'sharpe_limit': 'sharpe_ratio_threshold', 'r_sq_limit': 'r_squared_threshold',
            'z_entry': 'z_entry', 'z_exit': 'z_exit',
            'hl_multiplier': 'hl_multiplier', 'commission_pct': 'commission_pct',
        }
        for k, csv_key in mapping.items():
            if csv_key in df.index:
                val = str(df.loc[csv_key, 'Value']).replace(',', '.')
                res[k] = float(val)
            else:
                res[k] = defaults[k]
        return res
    except:
        return defaults

def calculate_mdd(equity_series):
    if len(equity_series) < 1:
        return 0.0
    peaks = np.maximum.accumulate(equity_series)
    drawdowns = (peaks - equity_series) / peaks
    return drawdowns.max() * 100

def safe_z(p1, p2, beta, intercept, spread_vol):
    """Z-score с защитой от деления на ноль"""
    if spread_vol == 0 or pd.isna(spread_vol):
        return np.nan
    return (p1 - beta * p2 - intercept) / spread_vol


def run_oos_backtest_continuous(is_results_path, settings_path, output_file="DATA/oos_results.xlsx"):
    stg = load_settings(settings_path)
    if not os.path.exists(is_results_path):
        print(f"Қате: Файл {is_results_path} табылмады.")
        return

    is_df = pd.read_excel(is_results_path, sheet_name='Pairs_Results')
    is_df['OOS_Start_Dt'] = pd.to_datetime(is_df['Month_OOS'], dayfirst=True)
    is_df = is_df.sort_values(['OOS_Start_Dt', 'Iteration'])

    start_date  = is_df['OOS_Start_Dt'].min()
    oos_end_date = is_df['OOS_Start_Dt'].max() + pd.offsets.MonthEnd(1)

    end_date = ALL_LOCAL_PRICES.index[-1]

    all_tickers = list(set(is_df['Stock_1'].tolist() + is_df['Stock_2'].tolist()))

    print(f"Жергілікті базадан бағалар алу, {len(all_tickers)} тикер...")
    clean_tickers = [t.replace('.', '-') for t in all_tickers]
    valid_tickers = [t for t in clean_tickers if t in ALL_LOCAL_PRICES.columns]
    prices = ALL_LOCAL_PRICES.loc[start_date:end_date, valid_tickers].ffill()

    if prices.empty:
        print("Қате: Бұл кезең үшін деректер табылмады.")
        return

    print(f"OOS кезеңі:      {start_date.date()} — {oos_end_date.date()}")
    print(f"Trailing кезеңі: {oos_end_date.date()} — {end_date.date()}")

    active_trades = {}
    banned_pairs = {}
    closed_trades = []
    initial_capital = stg['initial_capital']
    commission_pct  = stg['commission_pct']
    daily_equity = []
    daily_active_counts = []
    all_days = prices.index

    for i, current_date in enumerate(all_days):

        in_trailing = current_date > oos_end_date

        unrealized_pnl = 0
        for pk, trade in active_trades.items():
            if trade['Ticker_1'] not in prices.columns or trade['Ticker_2'] not in prices.columns:
                continue
            p1 = prices.loc[current_date, trade['Ticker_1']]
            p2 = prices.loc[current_date, trade['Ticker_2']]
            if pd.isna(p1) or pd.isna(p2):
                continue
            if trade['Spread_Type'] == "Long Spread":
                unrealized_pnl += (p1 - trade['Price_In_1']) * trade['Qty_1'] + \
                                   (trade['Price_In_2'] - p2) * trade['Qty_2']
            else:
                unrealized_pnl += (trade['Price_In_1'] - p1) * trade['Qty_1'] + \
                                   (p2 - trade['Price_In_2']) * trade['Qty_2']

        realized_pnl = sum(t.get('Total_PnL_USD', 0) for t in closed_trades)
        current_equity = initial_capital + realized_pnl + unrealized_pnl
        daily_equity.append(current_equity)
        daily_active_counts.append(len(active_trades))

        mode_tag = "[TRAIL]" if in_trailing else "[OOS]  "
        sys.stdout.write(f"\r{mode_tag} {current_date.date()} | Ашық: {len(active_trades)} | Тыйым: {len(banned_pairs)} | Капитал: {current_equity:.2f}")
        sys.stdout.flush()

        z_ex_lvl = stg.get('z_exit', 0.5)

        to_unban = []
        for pk, bp in banned_pairs.items():
            t1, t2 = bp['ticker_1'], bp['ticker_2']
            if t1 not in prices.columns or t2 not in prices.columns:
                to_unban.append(pk)
                continue
            p1b = prices.loc[current_date, t1]
            p2b = prices.loc[current_date, t2]
            if pd.isna(p1b) or pd.isna(p2b):
                continue
            zb = safe_z(p1b, p2b, bp['beta'], bp['intercept'], bp['spread_vol'])
            if not pd.isna(zb) and abs(zb) <= z_ex_lvl:
                to_unban.append(pk)
        for pk in to_unban:
            if pk in banned_pairs:
                del banned_pairs[pk]

        to_close = []
        hl_mult  = stg.get('hl_multiplier', 2.5)

        for pair_key, trade in active_trades.items():
            s1, s2 = trade['Ticker_1'], trade['Ticker_2']
            if s1 not in prices.columns or s2 not in prices.columns:
                to_close.append(pair_key)
                continue
            p1, p2 = prices.loc[current_date, s1], prices.loc[current_date, s2]
            if pd.isna(p1) or pd.isna(p2):
                continue

            z = safe_z(p1, p2, trade['Beta'], trade['Intercept'], trade['Spread_Vol'])
            if pd.isna(z):
                continue

            is_conv = (trade['Spread_Type'] == "Long Spread" and z >= -z_ex_lvl) or \
                      (trade['Spread_Type'] == "Short Spread" and z <= z_ex_lvl)
            duration = (current_date - pd.to_datetime(trade['Entry_Date'])).days
            is_time_out = duration > (trade.get('Half_Life', 10) * hl_mult)

            if is_conv or is_time_out:
                q1, q2 = trade['Qty_1'], trade['Qty_2']
                if trade['Spread_Type'] == "Long Spread":
                    trade_pnl = (p1 - trade['Price_In_1']) * q1 + (trade['Price_In_2'] - p2) * q2
                else:
                    trade_pnl = (trade['Price_In_1'] - p1) * q1 + (p2 - trade['Price_In_2']) * q2

                commission = commission_pct * (
                    trade['Price_In_1'] * q1 + trade['Price_In_2'] * q2 +
                    p1 * q1 + p2 * q2
                )
                trade_pnl -= commission
                reason = "Zero_Cross" if is_conv else "Time_Exit"

                if is_time_out and not is_conv:
                    banned_pairs[pair_key] = {
                        'beta': trade['Beta'], 'intercept': trade['Intercept'],
                        'spread_vol': trade['Spread_Vol'],
                        'ticker_1': trade['Ticker_1'], 'ticker_2': trade['Ticker_2']
                    }

                closed_trades.append({
                    **trade,
                    "Year": current_date.year,
                    "Price_Out_1": round(p1, 2), "Price_Out_2": round(p2, 2),
                    "Commission_USD": round(commission, 4),
                    "Total_PnL_USD": round(trade_pnl, 2),
                    "Total_PnL_Pct": round((trade_pnl / initial_capital) * 100, 4),
                    "Exit_Date": current_date.date(), "Exit_Reason": reason,
                    "Exit_Z_Score": round(z, 2), "Duration_Days": duration,
                    "Trailing": in_trailing
                })
                to_close.append(pair_key)

        for pk in to_close:
            if pk in active_trades:
                del active_trades[pk]

        if not in_trailing:
            curr_month_start = current_date.replace(day=1)
            iter_pairs = is_df[is_df['OOS_Start_Dt'] == curr_month_start]
            z_ent_lvl = stg.get('z_entry', 2.0)

            if not iter_pairs.empty and len(active_trades) < stg['max_open_pairs']:
                limit_per_pair = current_equity / stg['max_open_pairs']
                for _, row in iter_pairs.sort_values('Sharpe_IS', ascending=False).iterrows():
                    if len(active_trades) >= stg['max_open_pairs']:
                        break
                    pair_key = f"{row['Stock_1']}-{row['Stock_2']}"
                    if pair_key in active_trades:
                        continue
                    if pair_key in banned_pairs:
                        continue
                    if row['Stock_1'] not in prices.columns or row['Stock_2'] not in prices.columns:
                        continue

                    p1, p2 = prices.loc[current_date, row['Stock_1']], prices.loc[current_date, row['Stock_2']]
                    if pd.isna(p1) or pd.isna(p2):
                        continue

                    z = safe_z(p1, p2, row['Hedge_Ratio'], row['Intercept'], row['Spread_Vol'])
                    if pd.isna(z):
                        continue

                    if abs(z) > z_ent_lvl:
                        beta = row['Hedge_Ratio']
                        t_type = "Short Spread" if z > z_ent_lvl else "Long Spread"
                        qty1 = limit_per_pair / (p1 + abs(beta) * p2)
                        qty2 = qty1 * abs(beta)

                        active_trades[pair_key] = {
                            "Iteration": row['Iteration'], "Sector": row['Sector'], "Pair": pair_key,
                            "Entry_Date": current_date.date(), "Spread_Type": t_type,
                            "Ticker_1": row['Stock_1'], "Price_In_1": round(p1, 2), "Qty_1": round(qty1, 4),
                            "Ticker_2": row['Stock_2'], "Price_In_2": round(p2, 2), "Qty_2": round(qty2, 4),
                            "Beta": beta, "Intercept": row['Intercept'],
                            "Spread_Vol": row['Spread_Vol'], "Half_Life": row.get('Half_Life', 10),
                            "Year": current_date.year
                        }

        if in_trailing and len(active_trades) == 0:
            print(f"\n[INFO] Барлық позициялар жабылды {current_date.date()}, trailing аяқталды.")
            all_days = all_days[:i+1]
            daily_equity = daily_equity[:i+1]
            daily_active_counts = daily_active_counts[:i+1]
            break

    last_date = all_days[-1]
    open_positions = []
    for pair_key, trade in active_trades.items():
        s1, s2 = trade['Ticker_1'], trade['Ticker_2']
        p1 = prices.loc[last_date, s1] if s1 in prices.columns else np.nan
        p2 = prices.loc[last_date, s2] if s2 in prices.columns else np.nan
        q1, q2 = trade['Qty_1'], trade['Qty_2']

        if not pd.isna(p1) and not pd.isna(p2):
            if trade['Spread_Type'] == "Long Spread":
                unrealized = (p1 - trade['Price_In_1']) * q1 + (trade['Price_In_2'] - p2) * q2
            else:
                unrealized = (trade['Price_In_1'] - p1) * q1 + (p2 - trade['Price_In_2']) * q2
        else:
            unrealized = np.nan

        z = safe_z(p1, p2, trade['Beta'], trade['Intercept'], trade['Spread_Vol']) if not pd.isna(p1) else np.nan
        duration = (last_date - pd.to_datetime(trade['Entry_Date'])).days

        open_positions.append({
            **trade,
            "Last_Price_1": round(p1, 2) if not pd.isna(p1) else None,
            "Last_Price_2": round(p2, 2) if not pd.isna(p2) else None,
            "Last_Z_Score": round(z, 2) if not pd.isna(z) else None,
            "Duration_Days": duration,
            "Unrealized_PnL_USD": round(unrealized, 2) if not pd.isna(unrealized) else None,
            "Unrealized_PnL_Pct": round((unrealized / initial_capital) * 100, 4) if not pd.isna(unrealized) else None,
        })

    df_open = pd.DataFrame(open_positions) if open_positions else pd.DataFrame()

    print("\n")
    if not closed_trades:
        print("Жабық мәмілелер болмады.")
        if not df_open.empty:
            print(f"Кезең соңындағы ашық позициялар: {len(df_open)}")
        return

    df_t = pd.DataFrame(closed_trades)

    yearly = df_t.groupby('Year').agg(
        PnL_USD=('Total_PnL_USD', 'sum'), PnL_Pct=('Total_PnL_Pct', 'sum'),
        Trades=('Pair', 'count'),
    ).reset_index()

    yearly['Yearly_PF'] = 0.0
    for idx, r_y in yearly.iterrows():
        yr_d = df_t[df_t['Year'] == r_y['Year']]
        gp = yr_d[yr_d['Total_PnL_USD'] > 0]['Total_PnL_USD'].sum()
        gl = abs(yr_d[yr_d['Total_PnL_USD'] < 0]['Total_PnL_USD'].sum())
        yearly.at[idx, 'Yearly_PF'] = round(gp / gl if gl != 0 else float('inf'), 2)

    equity_series = pd.Series(daily_equity, index=all_days[:len(daily_equity)])
    daily_returns = equity_series.pct_change().dropna()
    strat_sharpe = (daily_returns.mean() / daily_returns.std() * np.sqrt(252)) if daily_returns.std() != 0 else 0
    downside_returns = daily_returns[daily_returns < 0]
    downside_std = downside_returns.std()
    strat_sortino = (daily_returns.mean() / downside_std * np.sqrt(252)) if downside_std != 0 else 0

    yearly_sharpe_map = daily_returns.groupby(daily_returns.index.year).apply(
        lambda x: (x.mean() / x.std() * np.sqrt(252)) if x.std() != 0 else 0)
    yearly_mdd_map = equity_series.groupby(equity_series.index.year).apply(calculate_mdd)

    yearly = yearly.merge(yearly_mdd_map.rename('MDD_Pct'), left_on='Year', right_index=True)
    yearly = yearly.merge(yearly_sharpe_map.rename('Yearly_Sharpe'), left_on='Year', right_index=True)

    real_util = (sum(daily_active_counts) / len(daily_active_counts) / stg['max_open_pairs']) * 100
    total_mdd = calculate_mdd(equity_series)
    total_ret = ((daily_equity[-1] / initial_capital) - 1) * 100
    days_total = (equity_series.index[-1] - equity_series.index[0]).days
    years_total = days_total / 365.25
    cagr = ((daily_equity[-1] / initial_capital)**(1/years_total) - 1) * 100 if years_total > 0 else 0

    gp_total = df_t[df_t['Total_PnL_USD'] > 0]['Total_PnL_USD'].sum()
    gl_total = abs(df_t[df_t['Total_PnL_USD'] < 0]['Total_PnL_USD'].sum())
    profit_factor = round(gp_total / gl_total if gl_total != 0 else float('inf'), 2)
    total_commission = df_t['Commission_USD'].sum() if 'Commission_USD' in df_t.columns else 0.0
    open_unrealized = df_open['Unrealized_PnL_USD'].sum() if not df_open.empty else 0.0

    trailing_count = int(df_t.get('Trailing', pd.Series(False)).sum()) if 'Trailing' in df_t.columns else 0

    summary_data = pd.DataFrame({
        "Metric": ["Total Return (%)", "Annual Return (CAGR %)", "Max Drawdown (%)", "Profit Factor",
                   "Win Rate (%)", "Sharpe Ratio", "Sortino Ratio", "Avg Duration", "Capital Utilization (%)",
                   "Total Commission (USD)", "Closed in Trailing", "Open Positions (EOD)", "Unrealized PnL (USD)"],
        "Value": [
            round(total_ret, 3), round(cagr, 2), round(total_mdd, 2), profit_factor,
            round((df_t['Total_PnL_USD'] > 0).mean() * 100, 2), round(strat_sharpe, 2), round(strat_sortino, 2),
            round(df_t['Duration_Days'].mean(), 1), round(real_util, 1),
            round(total_commission, 2), trailing_count, len(df_open), round(open_unrealized, 2)
        ]
    })

    ex_an = df_t.groupby('Exit_Reason').agg(
        Pair=('Pair', 'count'), PnL_Pct=('Total_PnL_Pct', 'mean')
    ).reset_index()

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_t.to_excel(writer, sheet_name='OOS_Trades', index=False)
        yearly.to_excel(writer, sheet_name='Yearly_Stats', index=False)
        summary_data.to_excel(writer, sheet_name='Summary', index=False)
        ex_an.to_excel(writer, sheet_name='Exit_Analysis', index=False)
        if not df_open.empty:
            df_open.to_excel(writer, sheet_name='Open_Positions', index=False)

    print("="*95)
    print(f"{' СТРАТЕГИЯНЫҢ ҚОРЫТЫНДЫ СТАТИСТИКАСЫ (OOS + TRAILING) ':-^95}")
    print("\n[ SUMMARY METRICS ]")
    print("-" * 45)
    for _, r in summary_data.iterrows():
        print(f"{r['Metric']:<28} | {r['Value']}")
    print("-" * 45)

    print("\n[ YEARLY PERFORMANCE ]")
    h = f"{'Year':<6} | {'PnL $':>10} | {'PnL %':>8} | {'MDD %':>8} | {'Sharpe':>8} | {'Trades':>7} | {'PF':>6}"
    print(h)
    print("-" * len(h))
    for _, r in yearly.iterrows():
        print(f"{int(r['Year']):<6} | {r['PnL_USD']:>10.2f} | {r['PnL_Pct']:>8.2f} | {r['MDD_Pct']:>8.2f} | "
              f"{r['Yearly_Sharpe']:>8.2f} | {int(r['Trades']):>7} | {r['Yearly_PF']:>6.2f}")

    print("\n[ EXIT ANALYSIS ]")
    print("-" * 50)
    for _, r in ex_an.iterrows():
        print(f"{r['Exit_Reason']:<15} | Pairs: {int(r['Pair']):<4} | Avg PnL %: {r['PnL_Pct']:>10.4f}")

    if not df_open.empty:
        print(f"\n[ OPEN POSITIONS AT END OF PERIOD: {len(df_open)} ]")
        print("-" * 60)
        total_unr = df_open['Unrealized_PnL_USD'].sum()
        print(f"  Жалпы реализацияланбаған PnL: {total_unr:>10.2f} USD ({total_unr/initial_capital*100:.2f}%)")
        print(f"  {'Жұп':<25} | {'Кіру':>12} | {'Unrealized $':>13} | {'Z':>7} | {'Күн':>6}")
        print("-" * 60)
        for _, r in df_open.iterrows():
            print(f"  {r['Pair']:<25} | {str(r['Entry_Date']):>12} | {r['Unrealized_PnL_USD']:>13.2f} | {r['Last_Z_Score']:>7.2f} | {int(r['Duration_Days']):>6}")

    print("\n" + "="*95)
    print(f"OOS нәтижелері сақталды: {output_file}")


run_oos_backtest_continuous("DATA/is_results.xlsx", "CSV/settings.csv")

In [ ]:
import shutil
import os
import pandas as pd
from dateutil.relativedelta import relativedelta



def save_run_snapshot(
    settings_path="CSV/settings.csv",
    schedule_path="CSV/schedule.csv",
    is_results_path="DATA/is_results.xlsx",
    oos_results_path="DATA/oos_results.xlsx",
    base_dir="DATA"
):
    scenario = "run"
    try:
        df_s = pd.read_csv(settings_path, delimiter=';', index_col='Parameter', dtype=str)
        if 'scenario' in df_s.index:
            scenario = str(df_s.loc['scenario', 'Value']).strip()
    except:
        pass

    is_window = "?"
    try:
        sched = pd.read_csv(schedule_path, delimiter=';')
        row0 = sched.iloc[0]
        is_start = pd.to_datetime(row0['IS_Start'], dayfirst=True)
        is_end   = pd.to_datetime(row0['IS_End'],   dayfirst=True)
        rd = relativedelta(is_end, is_start)
        is_window = str(rd.years * 12 + rd.months + 1)
    except:
        pass

    oos_range = "unknown"
    try:
        is_df = pd.read_excel(is_results_path, sheet_name='Pairs_Results')
        dates = pd.to_datetime(is_df['Month_OOS'], dayfirst=True)
        oos_range = f"{dates.min().year}-{dates.max().year}"
    except:
        pass

    folder_name = f"{oos_range}_{is_window}_{scenario}"
    run_dir = os.path.join(base_dir, "STAT_z", folder_name)

    if os.path.exists(run_dir):
        i = 2
        while os.path.exists(f"{run_dir}_{i}"):
            i += 1
        run_dir = f"{run_dir}_{i}"

    os.makedirs(run_dir, exist_ok=True)

    copied = []
    for src in [settings_path, is_results_path, oos_results_path]:
        if os.path.exists(src):
            dst = os.path.join(run_dir, os.path.basename(src))
            shutil.copy2(src, dst)
            copied.append(os.path.basename(src))
        else:
            print(f"[!] Табылмады: {src}")

    print(f"[OK] Сурет сақталды: {run_dir}")
    print(f"     Файлдар: {', '.join(copied)}")

save_run_snapshot()
